# Build a Reasoning Model: DeepSeek-R1 from Scratch

This notebook builds a reasoning model the way the **DeepSeek-R1 paper**
(arXiv:2501.12948, Figure 2) does — as an explicit, multi-stage pipeline — but
using tiny, intuitive, from-scratch PyTorch on one toy task you can run on a
laptop CPU: **1-digit integer addition**.

We track a single number the whole way down the pipeline: **pass-rate**, the
fraction of random `a+b` prompts the model answers correctly. Watch it climb
from ~0% to most of the way to 100% as we add each ingredient of the recipe.

### The map: R1 paper stage -> this notebook -> checkpoint

| R1 paper stage (Fig. 2) | Section here | Checkpoint written |
|---|---|---|
| *(motivation)* | 0. Foundations: reasoning changes the output contract | - |
| DeepSeek-V3-Base | 1. Base model (a plain next-token predictor) | - |
| **R1-Zero**: pure RL (GRPO) on the base | 2. R1-Zero path: RL from verifiable rewards | `nb2_r1_zero.pt` |
| Cold-start SFT | 3. Cold-start SFT (teach the think/answer format) | `nb2_cold_start.pt` |
| Reasoning-oriented RL (GRPO) | 4. Reasoning RL: pass-rate climbs | `nb2_after_grpo.pt` |
| Rejection-sampling SFT | 5. Amplify: the model as a data factory | `nb3_after_reject_sft.pt` |
| Distillation | 6. Compress: pour the skill into a smaller student | `nb3_student.pt` |
| *(payoff)* | 7. Recap: one metric climbed through every stage | - |

All the shared machinery (tokenizer, `TinyLM`, reward, sampling, `pass_rate`)
lives in **`r1_toy.py`** and is imported as `R`.

> **Marker note.** In Section 0 we use readable `<think>...</think><answer>...</answer>`
> tags, which a tiny char model learns fine for plain supervised training. From
> Section 2 on (the RL stages) we switch to compact one-character markers:
> `T` stands in for `<think>`, `A` for `<answer>`, `.` ends the sequence — e.g.
> `3+4=T3+4 A7.`. GRPO needs sampling *variety within a group* to get a learning
> signal; the compact alphabet keeps enough entropy for RL to work. Same idea,
> RL-friendly alphabet. (See the header of `r1_toy.py`.)

In [1]:
import os, sys, random, re, copy
import torch, torch.nn as nn, torch.nn.functional as F

sys.path.insert(0, os.getcwd())   # so `import r1_toy` resolves in nbconvert
torch.manual_seed(0); random.seed(0)
print('setup ready | torch', torch.__version__)

setup ready | torch 2.13.0


## 0. Foundations - a reasoning model changes the output contract

A **vanilla** LLM maps a question straight to an answer:

```
question  ->  answer
```

A **reasoning** LLM changes the *output contract*: it first emits a chain of
intermediate tokens - a *reasoning chain* / *scratchpad* - and only then commits
to an answer:

```
question  ->  reasoning chain  ->  answer
```

Why does that extra chain help? Two intuitions:

1. **Decomposition.** A hard problem is split into easier sub-steps the model
   has seen before (`17x24 = 17x20 + 17x4 = ...`).
2. **More compute = more thinking.** Each generated token is another forward
   pass. A model that answers in one token gets one pass to decide; a model that
   writes 30 reasoning tokens first gets ~30 extra passes of computation *before*
   committing. Reasoning tokens literally buy computation. This is why
   chain-of-thought is called **test-time compute**.

Below we turn intuition #2 into a number you can measure.

### 0a. The *shape* of the trick (optional GPT-2 demo)

The classic prompt-only trick: append *"Let's think step by step."* and watch a
model lay out intermediate steps instead of blurting an answer. GPT-2 (124M) is
too weak to reliably get the arithmetic *right*, so this only shows the *shape*
of the behavior - the measurable proof is the next demo.

This cell is **off by default** so the notebook runs fast and offline; flip
`RUN_GPT2_DEMO = True` to download `gpt2` and see it live.

In [2]:
RUN_GPT2_DEMO = False  # flip to True to download gpt2 (124M) and see the 'shape' live

if RUN_GPT2_DEMO:
    try:
        from transformers import pipeline, set_seed
        set_seed(0)
        gen = pipeline('text-generation', model='gpt2', device=-1)
        for tag, prompt, n in [
            ('DIRECT', 'Q: If I have 3 apples and buy 2 more, how many apples do I have?\nA:', 20),
            ('CHAIN-OF-THOUGHT', "Q: If I have 3 apples and buy 2 more, how many apples do I have?\nA: Let's think step by step.", 60),
        ]:
            out = gen(prompt, max_new_tokens=n, do_sample=False)[0]['generated_text']
            print('-' * 60); print(tag); print('-' * 60); print(out)
    except Exception as e:
        print('GPT-2 demo skipped (load failed):', e)
else:
    print('GPT-2 shape demo skipped (set RUN_GPT2_DEMO=True to run it live).')

GPT-2 shape demo skipped (set RUN_GPT2_DEMO=True to run it live).


### 0b. The measurable proof: does a scratchpad help?

Now we *measure* intuition #2. Task: **1-digit addition**. We train two identical
tiny char-level models, changing only the target format:

- **Direct model** must answer immediately: `3+4=<answer>7</answer>.`
- **Scratchpad model** gets room to "think" first: `3+4=<think>3+4</think><answer>7</answer>.`

Same architecture, same data, same steps. The only difference is whether the
model may emit intermediate tokens before the answer. Then we compare pass-rate.

In [3]:
torch.manual_seed(0); random.seed(0)

# Char vocab covering digits, operator, and the <think>/<answer> tag characters.
B_VOCAB = sorted(set(list('0123456789+=. ') + list('<>thinkanswer/')))
b_stoi = {c: i for i, c in enumerate(B_VOCAB)}
b_itos = {i: c for c, i in b_stoi.items()}
BV = len(B_VOCAB); B_PAD = b_stoi[' ']; B_EOS = b_stoi['.']; B_BLOCK = 48
b_enc = lambda s: [b_stoi[c] for c in s]
b_dec = lambda ids: ''.join(b_itos[i] for i in ids)

def direct_example(a, b):  return f'{a}+{b}=<answer>{a + b}</answer>.'
def scratch_example(a, b): return f'{a}+{b}=<think>{a}+{b}</think><answer>{a + b}</answer>.'

print('direct    :', direct_example(3, 4))
print('scratchpad:', scratch_example(3, 4))
print('vocab size:', BV, '| block:', B_BLOCK)

direct    : 3+4=<answer>7</answer>.
scratchpad: 3+4=<think>3+4</think><answer>7</answer>.
vocab size: 27 | block: 48


In [4]:
class DemoLM(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, block=B_BLOCK):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d, h, dim_feedforward=4 * d,
                                       batch_first=True, activation='gelu')
            for _ in range(L)])
        self.head = nn.Linear(d, V)

    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        for blk in self.blocks:
            h = blk(h, src_mask=mask)
        return self.head(h)


def b_pad_batch(strings):
    ids = [b_enc(s) for s in strings]
    L = max(len(x) for x in ids)
    return torch.tensor([x + [B_PAD] * (L - len(x)) for x in ids])


@torch.no_grad()
def b_generate(model, prompt, max_new=40):
    x = torch.tensor([b_enc(prompt)])
    for _ in range(max_new):
        nxt = torch.argmax(model(x[:, -B_BLOCK:])[:, -1, :], -1, keepdim=True)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == B_EOS:
            break
    return b_dec(x[0].tolist())


def b_pass_rate(model, n=200):
    correct = 0
    for _ in range(n):
        a, b = random.randint(0, 9), random.randint(0, 9)
        out = b_generate(model, f'{a}+{b}=')
        m = re.search(r'<answer>(\d+)</answer>', out)
        if m and int(m.group(1)) == a + b:
            correct += 1
    return correct / n


def train_demo(make_example, steps=600, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    examples = [make_example(a, b) for a in range(10) for b in range(10)]
    model = DemoLM(BV)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for _ in range(steps):
        x = b_pad_batch(random.choices(examples, k=32))
        logits = model(x[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, BV), x[:, 1:].reshape(-1),
                               ignore_index=B_PAD)
        opt.zero_grad(); loss.backward(); opt.step()
    return model

print('helpers ready')

helpers ready


In [5]:
print('training DIRECT model (no scratchpad)...')
direct_model = train_demo(direct_example)
print('training SCRATCHPAD model (think then answer)...')
scratch_model = train_demo(scratch_example)

direct_pr = b_pass_rate(direct_model)
scratch_pr = b_pass_rate(scratch_model)

print('-' * 60)
print('sample direct    :', b_generate(direct_model, '7+8='))
print('sample scratchpad:', b_generate(scratch_model, '7+8='))
print('-' * 60)
print(f'{"model":<24}{"pass-rate":>10}')
print(f'{"direct (answer only)":<24}{direct_pr:>9.0%}')
print(f'{"scratchpad (think 1st)":<24}{scratch_pr:>9.0%}')
print('-' * 60)
print(f'scratchpad advantage: {(scratch_pr - direct_pr) * 100:+.0f} points')

training DIRECT model (no scratchpad)...


training SCRATCHPAD model (think then answer)...


------------------------------------------------------------
sample direct    : 7+8=<answer>15</answer>.
sample scratchpad: 7+8=<think>7+8</think><answer>15</answer>.
------------------------------------------------------------
model                    pass-rate
direct (answer only)          88%
scratchpad (think 1st)        95%
------------------------------------------------------------
scratchpad advantage: +7 points


In [6]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3.2))
bars = ax.bar(['direct\n(answer only)', 'scratchpad\n(think first)'],
              [direct_pr * 100, scratch_pr * 100], color=['#bbbbbb', '#2c7fb8'])
ax.set_ylabel('pass-rate (%)'); ax.set_ylim(0, 100)
ax.set_title('Does an intermediate scratchpad help?\n1-digit addition, tiny char model')
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2,
            f'{b.get_height():.0f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

/var/folders/5l/y8s3fc655417629rqwgxkhx80000gn/T/ipykernel_80221/3050714067.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Takeaway (Foundations).** Same model, same data, same compute - the only
change is letting the model emit intermediate tokens before answering, and the
pass-rate goes up. That is a local, reproducible proof that reasoning tokens buy
computation, and computation buys accuracy. Everything below builds a real
training pipeline around exactly this idea.

## 1. Base model - the DeepSeek-V3-Base analog

**Paper stage: DeepSeek-V3-Base.** Every reasoning model starts life as a plain
**next-token predictor**: predict the next token, over and over, on a big
corpus. That is *pre-training*, and it gives the model fluency but no explicit
reasoning behavior yet. In our toy, the base is a ~104K-parameter char-level
transformer (`r1_toy.TinyLM`). Below we first show pre-training in miniature on
Shakespeare, then instantiate the base model that the R1-Zero and R1 paths will
both start from.

### 1a. Pre-training in one minute (next-token prediction)

The entire idea in one tiny run: a 2-layer char model, 200 steps, on a snippet
of Shakespeare. Real pre-training is the *same* next-token loss, just with
billions of tokens and parameters.

In [7]:
torch.manual_seed(0)
TEXT = (
    'To be, or not to be, that is the question:\n'
    'Whether tis nobler in the mind to suffer\n'
    'The slings and arrows of outrageous fortune,\n'
    'Or to take arms against a sea of troubles\n'
    'And by opposing end them.\n'
) * 20
chars = sorted(set(TEXT))
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in TEXT], dtype=torch.long)
SV, SBLOCK = len(chars), 32

class ShakespeareLM(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, block=SBLOCK):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d, h, dim_feedforward=4 * d,
                                       batch_first=True, activation='gelu')
            for _ in range(L)])
        self.head = nn.Linear(d, V)
    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        for blk in self.blocks:
            h = blk(h, src_mask=mask)
        return self.head(h)

smodel = ShakespeareLM(SV)
sopt = torch.optim.AdamW(smodel.parameters(), lr=3e-3)
for step in range(200):
    ix = torch.randint(0, len(data) - SBLOCK - 1, (16,))
    x = torch.stack([data[i:i + SBLOCK] for i in ix])
    y = torch.stack([data[i + 1:i + SBLOCK + 1] for i in ix])
    loss = F.cross_entropy(smodel(x).view(-1, SV), y.view(-1))
    sopt.zero_grad(); loss.backward(); sopt.step()
    if step % 40 == 0:
        print(f'step {step:3d}  loss {loss.item():.3f}')

@torch.no_grad()
def ssample(prompt='To be', n=80):
    x = torch.tensor([[stoi[c] for c in prompt]])
    for _ in range(n):
        nxt = torch.multinomial(F.softmax(smodel(x[:, -SBLOCK:])[:, -1, :], -1), 1)
        x = torch.cat([x, nxt], 1)
    return ''.join(itos[i.item()] for i in x[0])

print('-' * 60); print(ssample())

step   0  loss 3.422


step  40  loss 1.402


step  80  loss 0.669


step 120  loss 0.348


step 160  loss 0.262


------------------------------------------------------------
To be, or not the to be, that is question:
Whethether is nobler in the mind to suffer


### 1b. The base model for our task

Now we switch to the shared toy machinery in `r1_toy.py` and build the **base
model** for the addition task. Just like DeepSeek-V3-Base is pre-trained on a
huge corpus *before* any reasoning training, we give our `TinyLM` a short
next-token pre-training pass on the task's *language*: strings of the form
`{a}+{b}=T{a}+{b} A{c}.` where the answer `{c}` is a **random digit**, not
necessarily correct.

The effect: the base becomes **fluent in the format** - it reliably produces
well-formed `think ... answer ...` attempts - but it has **not** learned to
*add*. Its answers are essentially guesses, so its pass-rate sits near the
random floor. This is our V3-Base analog, and the single starting point for
**both** pipeline paths:
- **R1-Zero** (Section 2): pure RL straight on this base.
- **R1** (Sections 3-6): cold-start SFT, then RL, then amplify, then compress.

In [8]:
import r1_toy as R

torch.manual_seed(0); random.seed(0)
print('vocab:', R.VOCAB, '| size:', R.V)
print('formatted training string:', R.make_example(3, 4), ' (T=<think>, A=<answer>)')

# Pre-train the base on the task LANGUAGE (correct format, RANDOM answers):
# it learns to *speak* `... T ... A ... .` but not to add.
def noisy_example(a, b):
    return f'{a}+{b}=T{a}+{b} A{random.randint(0, 18)}.'

base = R.TinyLM()
opt = torch.optim.AdamW(base.parameters(), lr=3e-3)
for step in range(700):
    x = R.pad_batch([noisy_example(random.randint(0, 9), random.randint(0, 9))
                     for _ in range(32)])
    logits = base(x[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, R.V), x[:, 1:].reshape(-1),
                           ignore_index=R.PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 140 == 0:
        print(f'pretrain step {step:3d}  loss {loss.item():.3f}')

pr_base = R.pass_rate(base, n=200)
print('-' * 60)
print('base params        :', round(R.n_params(base), 1), 'K')
print('samples from base  :', [R.sample_text(base, p) for p in ['3+4=', '7+8=']])
print(f'PASS-RATE of base model: {pr_base:.0%}   (fluent format, cannot add -> the floor)')

vocab: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '=', 'T', 'A', ' ', '.'] | size: 16
formatted training string: 3+4=T3+4 A7.  (T=<think>, A=<answer>)
pretrain step   0  loss 2.874


pretrain step 140  loss 0.516


pretrain step 280  loss 0.527


pretrain step 420  loss 0.515


pretrain step 560  loss 0.510


------------------------------------------------------------
base params        : 103.6 K
samples from base  : ['3+4=T3+41417.', '7+8=T7+82.']
PASS-RATE of base model: 2%   (fluent format, cannot add -> the floor)


## 2. R1-Zero path - pure RL (GRPO) on the base

**Paper stage: DeepSeek-R1-Zero.** The paper's boldest experiment: skip
supervised fine-tuning entirely and run **reinforcement learning directly on the
base model**, using only *verifiable rewards* (is the answer correct? is the
format right?). Reasoning behavior *emerges* on its own - but the outputs are
**ugly**: poor readability, mixed languages, messy scratchpads. That ugliness is
exactly why the full R1 pipeline (Section 3 onward) adds a small cold-start SFT
before RL.

Here we run GRPO straight on our untrained base and watch reasoning structure
emerge from nothing but a reward function.

### GRPO in four lines of intuition

**Group Relative Policy Optimization.** For each prompt:

1. **Sample `G` completions** from the *current* model.
2. **Score each** with a rule-based reward (`r1_toy.reward`): a little credit for
   using the `T ... A ...` format, a lot for the correct answer.
3. **Group-relative advantage:** turn the `G` rewards into z-scores,
   `adv = (r - mean) / (std + eps)`. Above-average completions get positive
   advantage; below-average ones negative.
4. **REINFORCE update:** `loss = -(logprob * advantage).mean()` - push the policy
   toward better-than-average completions, away from worse.

**No critic, no value network, no human labels.** The other samples in the group
*are* the baseline - "the group is the baseline." The only external signal is a
Python function that can check the answer: **RL from verifiable rewards.**

In [9]:
def grpo_step(model, prompt_str, G=4):
    # one GRPO update for a single prompt; returns (loss, mean_reward, completions)
    prompt_ids = torch.tensor([R.encode(prompt_str)])
    logps, rewards, comps = [], [], []
    for _ in range(G):
        full, lp = R.sample_completion(model, prompt_ids)   # sample + log-probs
        comp = R.decode(full[0, len(prompt_ids[0]):].tolist())
        comps.append(comp); logps.append(lp)
        rewards.append(R.reward(prompt_str, comp))          # rule-based score
    r = torch.tensor(rewards)
    adv = (r - r.mean()) / (r.std() + 1e-6)                  # group-relative advantage
    loss = -(torch.cat(logps) * adv).mean()                 # REINFORCE with advantage
    return loss, r.mean().item(), comps

print('grpo_step defined (G completions -> z-score advantage -> REINFORCE)')

grpo_step defined (G completions -> z-score advantage -> REINFORCE)


In [10]:
# R1-Zero: GRPO straight on the base, no SFT first
torch.manual_seed(0); random.seed(0)
zero_model = copy.deepcopy(base)
opt = torch.optim.AdamW(zero_model.parameters(), lr=3e-4)

zero_reward_hist = []
for step in range(400):
    prompt, a, b = R.random_prompt()
    loss, mean_r, comps = grpo_step(zero_model, prompt, G=8)
    opt.zero_grad(); loss.backward(); opt.step()
    zero_reward_hist.append(mean_r)
    if step % 50 == 0:
        pr = R.pass_rate(zero_model, n=60)
        print(f'step {step:3d}  mean_r {mean_r:.2f}  pass-rate~{pr:.0%}   e.g. {prompt}{comps[0]}')

pr_zero = R.pass_rate(zero_model, n=200)
torch.save(zero_model.state_dict(), 'nb2_r1_zero.pt')
print('-' * 60)
print('R1-Zero sample outputs (note the ugliness / malformed scratchpads):')
for p in ['3+4=', '7+8=', '5+5=', '9+6=']:
    print(f'   {p}{R.sample_text(zero_model, p)}')
print('-' * 60)
print(f'PASS-RATE after pure RL (R1-Zero): {pr_zero:.0%}')
print('saved nb2_r1_zero.pt')

step   0  mean_r 0.06  pass-rate~2%   e.g. 6+6=T6+63A4.


step  50  mean_r 0.10  pass-rate~2%   e.g. 2+4=T2+41A14.


step 100  mean_r 0.35  pass-rate~3%   e.g. 0+8=T0+85A0.


step 150  mean_r 0.10  pass-rate~5%   e.g. 6+0=T6+03A13.


step 200  mean_r 0.10  pass-rate~10%   e.g. 5+6=T5+61A2.


step 250  mean_r 0.10  pass-rate~2%   e.g. 8+0=T8+30A6.


step 300  mean_r 0.23  pass-rate~5%   e.g. 7+1=T7+17A8.


step 350  mean_r 0.10  pass-rate~5%   e.g. 6+7=T6+76A5.


------------------------------------------------------------
R1-Zero sample outputs (note the ugliness / malformed scratchpads):
   3+4=3+4=T3+47A6.
   7+8=7+8=T7+87A14.
   5+5=5+5=T5+57A15.
   9+6=9+6=T9+67A12.
------------------------------------------------------------
PASS-RATE after pure RL (R1-Zero): 7%
saved nb2_r1_zero.pt


**Takeaway (R1-Zero).** With *no supervised reasoning data at all* - only a
reward function that can check answers - real **skill emerged**: starting from a
base that could only *guess*, pure RL pushed the pass-rate up off the floor and
the model began producing genuinely correct `T ... A ...` completions. That is
the striking R1-Zero result: reasoning improves from a verifier alone.

But pure RL is hard and unstable here - the gains are modest and the scratchpads
stay messy (garbled think slots, occasional malformed answers). In the real paper
R1-Zero showed the same pattern: strong emergent reasoning but poor readability
and language mixing. The fix is a *small* dose of supervised examples **before**
RL - the full **R1** path we build next. From here on we restart from the
**base** and take the cleaner route.

## 3. R1 path, step 1 - Cold-start SFT (teach the format)

**Paper stage: Cold-start SFT.** Before any RL, R1 fine-tunes the base on a small
set of high-quality examples that demonstrate the desired *reasoning format*.
This "cold start" locks in a clean, readable shape so the subsequent RL is stable
and the outputs are legible (fixing R1-Zero's ugliness). In our toy we
supervised-fine-tune on just **20** formatted examples - enough to teach the
`... T ... A ... .` shape, *not* enough to make the model good at addition yet.

In [11]:
# Restart from the base, then SFT on 20 formatted examples
torch.manual_seed(0); random.seed(0)
model = copy.deepcopy(base)

examples = R.all_examples(); random.shuffle(examples)
sft_data = examples[:20]
print('cold-start set (20 examples), first 5:')
for ex in sft_data[:5]:
    print('  ', ex)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(400):
    x = R.pad_batch(random.choices(sft_data, k=16))
    logits = model(x[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, R.V), x[:, 1:].reshape(-1),
                           ignore_index=R.PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 80 == 0:
        print(f'step {step:3d}  loss {loss.item():.3f}')

pr_coldstart = R.pass_rate(model, n=200)
torch.save(model.state_dict(), 'nb2_cold_start.pt')
print('-' * 60)
print('sample:', R.generate(model, '3+4='))
print('sample:', R.generate(model, '7+8='))
print(f'PASS-RATE after cold-start SFT: {pr_coldstart:.0%}')
print('saved nb2_cold_start.pt')

cold-start set (20 examples), first 5:
   2+3=T2+3 A5.
   0+8=T0+8 A8.
   1+1=T1+1 A2.
   0+7=T0+7 A7.
   4+8=T4+8 A12.
step   0  loss 0.517


step  80  loss 0.122


step 160  loss 0.096


step 240  loss 0.124


step 320  loss 0.114


------------------------------------------------------------
sample: 3+4=T3+41+4.
sample: 7+8=T7+87.
PASS-RATE after cold-start SFT: 8%
saved nb2_cold_start.pt


**What to notice.** Outputs now have the right *shape* - `T ... A ... .` -
but the answers are usually wrong. Twenty examples bought the **format**, not the
**skill**. The pass-rate is low. RL fixes that next - and because the format is
clean, the RL will be stable and the outputs readable (unlike R1-Zero).

## 4. R1 path, step 2 - Reasoning-oriented RL (GRPO)

**Paper stage: Reasoning-oriented RL.** Same GRPO as R1-Zero, but now starting
from the cleanly-formatted cold-start model. The reward mixes **accuracy** (is
the sum right?) and **format** (did it use the think/answer structure?); the
paper also adds a **language-consistency** reward to stop language mixing - we
skip that here since our toy alphabet is language-free, but it slots in as just
another term in the same `reward()` sum. No value net, no human labels: the group
is the baseline. Watch the pass-rate climb.

In [12]:
torch.manual_seed(0); random.seed(0)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
mean_reward_hist = []

for step in range(300):
    prompt, a, b = R.random_prompt()
    loss, mean_r, comps = grpo_step(model, prompt, G=4)
    opt.zero_grad(); loss.backward(); opt.step()
    mean_reward_hist.append(mean_r)
    if step % 30 == 0:
        pr = R.pass_rate(model, n=60)
        print(f'step {step:3d}  mean_r {mean_r:.2f}  pass-rate~{pr:.0%}   e.g. {prompt}{comps[0]}')

pr_grpo = R.pass_rate(model, n=200)
torch.save(model.state_dict(), 'nb2_after_grpo.pt')
print('-' * 60)
print(f'PASS-RATE after cold-start SFT : {pr_coldstart:.0%}')
print(f'PASS-RATE after reasoning RL   : {pr_grpo:.0%}')
print('saved nb2_after_grpo.pt')

step   0  mean_r 0.55  pass-rate~12%   e.g. 6+6=T6+6=T6.


step  30  mean_r 0.10  pass-rate~28%   e.g. 9+2=T9+2=A12.


step  60  mean_r 0.10  pass-rate~20%   e.g. 7+1=T7+1=A1.


step  90  mean_r 0.08  pass-rate~17%   e.g. 1+0=T1+0=A2.


step 120  mean_r 0.10  pass-rate~17%   e.g. 7+9=T7+97A17.


...

step 180  mean_r 1.10  pass-rate~27%   e.g. 6+3=T6+3=A9.


step 210  mean_r 0.10  pass-rate~22%   e.g. 2+1=T2+1=A5.


step 240  mean_r 0.10  pass-rate~13%   e.g. 5+6=T5+6=A12.


step 270  mean_r 0.10  pass-rate~22%   e.g. 6+1=T6+1=A1.


------------------------------------------------------------
PASS-RATE after cold-start SFT : 8%
PASS-RATE after reasoning RL   : 19%
saved nb2_after_grpo.pt


In [13]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
h = np.array(mean_reward_hist)
if len(h) >= 20:
    ax1.plot(np.convolve(h, np.ones(20) / 20, mode='valid'))
ax1.set_xlabel('GRPO step'); ax1.set_ylabel('mean reward (MA20)')
ax1.set_title('Reasoning RL: mean reward climbing')

bars = ax2.bar(['cold-start\nSFT', 'after\nGRPO'],
               [pr_coldstart * 100, pr_grpo * 100], color=['#bbbbbb', '#2c7fb8'])
ax2.set_ylabel('pass-rate (%)'); ax2.set_ylim(0, 100)
ax2.set_title('Pass-rate: before vs after RL')
for b in bars:
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 2,
             f'{b.get_height():.0f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

/var/folders/5l/y8s3fc655417629rqwgxkhx80000gn/T/ipykernel_80221/1196572663.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Takeaway (Reasoning RL).** We improved the *skill* with reinforcement
learning - no critic, no value network, no human labels - only a Python
`reward()` that can check an answer. Whenever you can *verify* an answer with
code (math, unit tests, a parser, a game score), you can turn that checker into a
reward and improve a model from its own samples. That is the core move behind
R1-style reasoning training.

## 5. R1 path, step 3 - Rejection-sampling SFT (amplify)

**Paper stage: Rejection-sampling SFT.** Use the RL'd model as a *data factory*:
generate several completions per prompt, **keep only the correct ones** (reject
the rest), and supervised-fine-tune on that filtered set. The model learns from
its own best outputs - **no new labels, no new reward** - and the pass-rate jumps.
R1 used this to turn a wobbly RL policy into clean, deployable data; it is also
the trick behind modern self-improvement loops.

In [14]:
# Step 1+2: generate completions per prompt, keep the first correct one
def is_correct(text, a, b):
    m = re.search(r'A(\d+)\.', text)
    return m is not None and int(m.group(1)) == a + b

torch.manual_seed(0); random.seed(0)
kept = []
for a in range(10):
    for b in range(10):
        for _ in range(12):                       # up to 12 tries per prompt
            out = R.sample_text(model, f'{a}+{b}=')
            if is_correct(out, a, b):
                kept.append(out); break

print(f'kept {len(kept)} / 100 prompts (correct, self-generated)')
for ex in kept[:5]:
    print('  ', ex)

kept 31 / 100 prompts (correct, self-generated)
   0+1=T0+1=A1.
   0+2=T0+2=A2.
   0+5=T0+5=A5.
   0+7=T0+7=A7.
   0+8=T0+8=A8.


In [15]:
# Step 3: SFT on the filtered, known-correct set
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(300):
    x = R.pad_batch(random.choices(kept, k=16))
    logits = model(x[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, R.V), x[:, 1:].reshape(-1),
                           ignore_index=R.PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 60 == 0:
        print(f'step {step:3d}  loss {loss.item():.3f}')

pr_reject = R.pass_rate(model, n=200)
torch.save(model.state_dict(), 'nb3_after_reject_sft.pt')
print('-' * 60)
print(f'PASS-RATE after reasoning RL   : {pr_grpo:.0%}')
print(f'PASS-RATE after rejection-SFT  : {pr_reject:.0%}')
print('saved nb3_after_reject_sft.pt')

step   0  loss 0.516


step  60  loss 0.154


step 120  loss 0.129


step 180  loss 0.104


step 240  loss 0.126


------------------------------------------------------------
PASS-RATE after reasoning RL   : 19%
PASS-RATE after rejection-SFT  : 28%
saved nb3_after_reject_sft.pt


**What to notice.** The model trained on *its own correct outputs* - no new
labels, no new reward - and the pass-rate rose. Filtering for correctness turns a
noisy policy into clean supervised data.

## 6. R1 path, step 4 - Distillation (compress)

**Paper stage: Distillation.** The paper's striking result: distilling a strong
reasoning model into a *small* one works **better** than running RL on the small
model directly - the teacher already did the hard exploration. We do the simplest
version:

- **Teacher** = our reject-SFT model (`d=64, L=2`, ~104K params).
- **Student** = half-width, single-layer (`d=32, L=1`, ~15K params, ~1/7 the size).
- **Objective** = KL divergence between student and teacher next-token
  distributions on *softened* logits (temperature `T=2`) - match the full
  distribution, not just the argmax.

In [16]:
teacher = model; teacher.eval()
student = R.TinyLM(d=32, L=1)   # half width, single layer

print('teacher params:', round(R.n_params(teacher), 1), 'K')
print('student params:', round(R.n_params(student), 1), 'K')
print('compression   :', f'{R.n_params(teacher) / R.n_params(student):.1f}x smaller')

# Distillation set: the teacher's own greedy completion for every prompt
distill_set = [R.generate(teacher, f'{a}+{b}=') for a in range(10) for b in range(10)]

opt = torch.optim.AdamW(student.parameters(), lr=3e-3)
T_temp = 2.0
for step in range(600):
    x = R.pad_batch(random.choices(distill_set, k=16))
    with torch.no_grad():
        t_logits = teacher(x[:, :-1]) / T_temp
    s_logits = student(x[:, :-1]) / T_temp
    loss = F.kl_div(F.log_softmax(s_logits, -1), F.softmax(t_logits, -1),
                    reduction='batchmean') * (T_temp ** 2)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f'step {step:3d}  KL {loss.item():.3f}')

pr_teacher = R.pass_rate(teacher, n=200)
pr_student = R.pass_rate(student, n=200)
torch.save(student.state_dict(), 'nb3_student.pt')
print('-' * 60)
print(f'teacher pass-rate ({R.n_params(teacher):.0f}K params): {pr_teacher:.0%}')
print(f'student pass-rate ({R.n_params(student):.0f}K params): {pr_student:.0%}')
print('saved nb3_student.pt')

teacher params: 103.6 K
student params: 14.5 K
compression   : 7.1x smaller


step   0  KL 114.246


step 100  KL 21.103


step 200  KL 7.898


step 300  KL 5.785


step 400  KL 3.580


step 500  KL 3.332


------------------------------------------------------------
teacher pass-rate (104K params): 28%
student pass-rate (15K params): 22%
saved nb3_student.pt


**What to notice.** A model with half the width and half the depth picked up
most of the teacher's skill from plain KL on logits. This is exactly why the real
R1 distilled checkpoints (Qwen-1.5B / 7B / 14B) punch above their weight: the
expensive RL exploration is done once by the teacher, then *copied* cheaply into
smaller students.

## 7. Recap - one metric climbed through every stage

Pass-rate across the whole R1 pipeline, on one toy task. The R1 path
(base -> cold-start -> RL -> reject-SFT -> distill) is the climbing arc; R1-Zero
is the pure-RL side branch that reached real skill but with ugly outputs.

In [17]:
stages = [
    ('base\n(untrained)',   pr_base,      '#dddddd'),
    ('R1-Zero\n(pure RL)',  pr_zero,      '#fdae6b'),
    ('cold-start\nSFT',     pr_coldstart, '#bbbbbb'),
    ('reasoning\nRL (GRPO)',pr_grpo,      '#7fcdbb'),
    ('reject-\nSFT',        pr_reject,    '#41b6c4'),
    ('distilled\nstudent',  pr_student,   '#2c7fb8'),
]

print(f'{"stage":<26}{"pass-rate":>10}')
print('-' * 36)
for name, pr, _ in stages:
    print(f'{name.replace(chr(10), " "):<26}{pr:>9.0%}')
print('-' * 36)
print(f'student is ~{R.n_params(teacher) / R.n_params(student):.0f}x smaller than the teacher')

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.5, 3.8))
bars = ax.bar([s[0] for s in stages], [s[1] * 100 for s in stages],
              color=[s[2] for s in stages])
ax.set_ylabel('pass-rate (%)'); ax.set_ylim(0, 100)
ax.set_title('1-digit addition pass-rate across the DeepSeek-R1 pipeline')
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2,
            f'{b.get_height():.0f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

stage                      pass-rate
------------------------------------
base (untrained)                 2%
R1-Zero (pure RL)                7%
cold-start SFT                   8%
reasoning RL (GRPO)             19%
reject- SFT                     28%
distilled student               22%
------------------------------------
student is ~7x smaller than the teacher


/var/folders/5l/y8s3fc655417629rqwgxkhx80000gn/T/ipykernel_80221/2934026541.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Takeaway - the whole recipe, one metric

| Paper stage | Section | What it adds |
|---|---|---|
| DeepSeek-V3-Base | 1 | Language (next-token prediction) |
| R1-Zero (pure RL) | 2 | Proof that reasoning *emerges* from verifiable rewards - but ugly |
| Cold-start SFT | 3 | **Format** (clean think -> answer shape) |
| Reasoning RL (GRPO) | 4 | **Skill** (correct answers, from verifiable rewards) |
| Rejection-sampling SFT | 5 | **Amplification** (train on own correct outputs) |
| Distillation | 6 | **Efficiency** (most of the skill, ~1/7 the size) |

One task, one metric, watched the whole way down the pipeline. A scratchpad
helped (Section 0); pure RL proved reasoning can emerge from a checker alone
(R1-Zero); cold-start gave it a clean shape; GRPO from verifiable rewards gave it
skill; rejection sampling amplified it; distillation compressed it. This is the
**exact same recipe** - GRPO + RL from verifiable rewards - that scales from this
104K-parameter toy to frontier reasoning models.